# Ollama model server on Colab, v3 (models saved in Google Drive)

**Every session:** Runtime > Change runtime type > **T4 GPU**, then Runtime > **Run all**.
First time: downloads the models into your Drive (slow). Later: loads them from Drive (no download).
The tunnel step prints a public URL. Paste it into your `.env` file as `COLAB_BASE_URL`.
Keep this tab open while you use the URL. The URL changes every session.

**To add another model later:** use the last cell (Step 9). It downloads the model into Drive and prints the line to add to `models.py`.

## Step 1: GPU check and connect Google Drive

In [ ]:
!nvidia-smi

from google.colab import drive
drive.mount('/content/drive')

## Step 2: Settings (the only cell you edit)
This list is your template. To keep a model available every session, add its Ollama tag here.
Find tags at ollama.com/library (click a model, then the Tags tab). Hugging Face GGUF models use `hf.co/<user>/<repo>:<quantization>`.
Rule of thumb for a free T4 (15 GB): models up to about 8B parameters in 4-bit (about 5 GB each).

In [ ]:
# Models are stored here, inside your Drive. The path MUST start with "/".
MODEL_DIR = "/content/drive/MyDrive/ollama_models"

# Add or remove models here. Names are the exact Ollama tags.
MODELS_TO_PULL = [
    "qwen2.5:7b",
    "hf.co/unsloth/Qwen3-4B-Instruct-2507-GGUF:Q4_K_M",
    "llama3.2:3b",
    "smollm2:1.7b",
]

assert MODEL_DIR.startswith("/"), "MODEL_DIR must start with /"

## Step 3: Install Ollama (needed every session; pciutils and lshw let it detect the GPU)

In [ ]:
!apt-get install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

## Step 4: Start the server, pointing at your Drive folder

In [ ]:
import os, subprocess, time, urllib.request

os.makedirs(MODEL_DIR, exist_ok=True)

env = os.environ.copy()
env["OLLAMA_MODELS"] = MODEL_DIR          # where models are stored (Drive)
env["OLLAMA_KEEP_ALIVE"] = "1h"           # keep models in GPU memory between requests
env["OLLAMA_MAX_LOADED_MODELS"] = "4"     # allow several models to stay loaded if they fit

subprocess.run(["pkill", "ollama"])       # stop any old server from an earlier run
time.sleep(2)
subprocess.Popen(["ollama", "serve"], env=env,
                 stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)

for _ in range(30):
    try:
        urllib.request.urlopen("http://localhost:11434")
        print("Server is ready")
        break
    except Exception:
        time.sleep(1)
else:
    print("Server did not start. Look inside ollama.log")

## Step 5: Download only the models that are missing from Drive

In [ ]:
import shutil

try:
    free_gb = shutil.disk_usage("/content/drive/MyDrive").free / 1e9
    print(f"Approx. free Drive space: {free_gb:.1f} GB")
except Exception:
    pass

installed = subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout

for tag in MODELS_TO_PULL:
    if tag in installed:
        print("Already in Drive, skipping:", tag)
    else:
        print("Downloading:", tag, "(can take several minutes, please wait)")
        get_ipython().system(f'ollama pull "{tag}" 2>&1 | tail -n 2')

print()
get_ipython().system("ollama list")

## Step 6: Warm-up test: cold start vs warm speed
The first call to a model loads it from Drive into the GPU, so it is slow (cold). The second call is the normal speed (warm).
Your evaluation should use the warm speed.

In [ ]:
def ask(tag):
    start = time.time()
    out = subprocess.run(["ollama", "run", tag, "Reply with one word: hello"],
                         capture_output=True, text=True)
    return out.stdout.strip().replace("\n", " ")[:40], time.time() - start

for tag in MODELS_TO_PULL:
    answer, cold = ask(tag)
    _, warm = ask(tag)
    print(f"{tag}\n   answer={answer!r}   cold={cold:.1f}s   warm={warm:.1f}s")

print()
print("Models currently in GPU memory (PROCESSOR should say GPU, not CPU):")
get_ipython().system("ollama ps")

## Step 7: Create the public URL (waits until the tunnel is really connected)

In [ ]:
import re

if not os.path.exists("cloudflared"):
    subprocess.run(["wget", "-q",
                    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
                    "-O", "cloudflared"])
    os.chmod("cloudflared", 0o755)

subprocess.run(["pkill", "cloudflared"])
time.sleep(1)
subprocess.Popen(
    "./cloudflared tunnel --protocol http2 --url http://localhost:11434 "
    "--http-host-header=localhost:11434 > tunnel.log 2>&1",
    shell=True)

url, connected = None, False
for _ in range(60):
    time.sleep(1)
    log = open("tunnel.log").read()
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", log)
    if match:
        url = match.group(0)
    if "Registered tunnel connection" in log:
        connected = True
        break

if url and connected:
    print("Tunnel is connected.")
    print("Your public URL:", url)
    print()
    print("Paste this line into your .env file on your laptop:")
    print(f"COLAB_BASE_URL={url}")
else:
    print("Tunnel NOT connected yet. Last lines of tunnel.log:")
    print("\n".join(open("tunnel.log").read().splitlines()[-12:]))

## Step 8: Final checks
1. The disk usage below should be several GB. That proves the models are saved in Drive.
2. **Test the URL from your phone** (mobile data is best): open your URL followed by `/api/tags`.
   You should see JSON listing your models.
Do not test the URL from inside Colab. A brand-new tunnel name often does not resolve there for a few minutes.

In [ ]:
get_ipython().system(f"du -sh {MODEL_DIR}")
print()
print("Open this on your phone:", (url or "<no url yet>") + "/api/tags")

## Step 9 (optional, any time): add one more model
1. Change `NEW_MODEL` to the Ollama tag you want (check the exact tag at ollama.com/library).
2. Run the cell. It downloads the model into Drive, tests it cold and warm, and prints the line to add to `COLAB_MODELS` in `models.py` on your laptop.
3. To make it download automatically in future sessions, also add the tag to `MODELS_TO_PULL` in Step 2.
The same tunnel URL serves the new model. No restart is needed.
Keep the total under about 15 GB of GPU memory. If the GPU is full, Ollama unloads idle models automatically, and the next call to an unloaded model is slow again.

In [ ]:
NEW_MODEL = "gemma3:4b"   # example only: change this, and check the exact tag at ollama.com/library

try:
    free_gb = shutil.disk_usage("/content/drive/MyDrive").free / 1e9
    print(f"Approx. free Drive space: {free_gb:.1f} GB")
except Exception:
    pass

print("Downloading", NEW_MODEL, "(can take several minutes)")
get_ipython().system(f'ollama pull "{NEW_MODEL}" 2>&1 | tail -n 2')

answer, cold = ask(NEW_MODEL)
_, warm = ask(NEW_MODEL)
print(f"answer={answer!r}   cold={cold:.1f}s   warm={warm:.1f}s")

safe_name = re.sub(r"[^a-z0-9.]+", "-", NEW_MODEL.lower()).strip("-")
print()
print("Add this line to COLAB_MODELS in models.py:")
print(f'    ("colab-{safe_name}", "{NEW_MODEL}"),')
print()
get_ipython().system("ollama ps")